# Ordered Logistic Regression Results for Adoption Predictors of Indigenous and Modern Knowledge in Rangeland Management Practices, Northern Kenya: Exploration with `mlcroissant`
This notebook provides a guided example for loading, exploring, and processing the FAIR² dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant --quiet

## 1. Data Loading
Load metadata and records from the FAIR² dataset using `mlcroissant`.

In [ ]:
import mlcroissant as mlc
import pandas as pd
import pprint

# Define the Croissant schema URL
croissant_url = "https://sen.science/doi/10.71728/senscience.y7m0-f273/fair2.json"

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata
print(f"{metadata.name}: {metadata.description}\n")
print(f"Dataset DOI: {metadata.identifier}")

## 2. Data Overview
Review available record sets, fields, and their IDs.

We will list each discovered record set and show a preview of the fields for inspection. All elements will be referenced using their `@id` for clarity and reproducibility.

In [ ]:
# List record sets and their fields by @id
record_sets = dataset.record_sets()
if not record_sets:
    print("No record sets detected in the Croissant schema. This may indicate the schema is metadata-only or uses references to external files.\n")
else:
    for record_set in record_sets:
        print(f"RecordSet @id: {record_set['@id']}")
        fields = record_set.get('field', [])
        if not fields:
            print('  (No fields listed)')
        else:
            for field in fields:
                if isinstance(field, dict):
                    fid = field.get('@id', str(field))
                    print(f"  Field @id: {fid}")
                else:
                    print(f"  Field @id: {field}")
        print()

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis.

(Note: Actual availability of record sets depends on the data package. If none are present, an explanatory message will be shown. Record sets and fields are always referenced by `@id`.)

In [ ]:
# For demonstration: extract data from all record sets
record_sets = dataset.record_sets()
dataframes = {}

if not record_sets:
    print('No record sets available for extraction.')
else:
    for rs in record_sets:
        rs_id = rs['@id']
        print(f"Loading records for RecordSet with @id: {rs_id}")
        records = list(dataset.records(record_set=rs_id))
        dataframes[rs_id] = pd.DataFrame(records)
        print(f"Fields: {list(dataframes[rs_id].columns)}\n")
    # Preview the first record set
    preview_rs_id = record_sets[0]['@id']
    print(f\"First 5 rows of RecordSet {preview_rs_id}:\")
    display(dataframes[preview_rs_id].head())

## 4. Exploratory Data Analysis (EDA)
Apply common data processing steps, such as filtering records, normalizing numeric fields, and grouping data. All columns and variables are referenced by their `@id`.

In [ ]:
# Choose a record set and numeric field (referenced by @id)
record_sets = dataset.record_sets()
if not record_sets:
    print('No record sets available for EDA.')
else:
    record_set_id = record_sets[0]['@id']
    df = dataframes[record_set_id]
    print(f"Working with record set: {record_set_id}")
    # List numeric-type fields (example: pick the first float/integer field if possible)
    numeric_field_id = None
    rs_meta = None
    # Find matching RecordSet metadata
    for rs in record_sets:
        if rs['@id'] == record_set_id:
            rs_meta = rs
            break

    # Identify a numeric field
    if rs_meta and 'field' in rs_meta:
        fields = rs_meta['field']
        # Ensure fields are dicts if possible
        for field in fields:
            if isinstance(field, dict) and 'dataType' in field:
                dtype = field['dataType']
                if dtype in ("schema:Integer", "schema:Float", "schema:Number"):
                    field_id = field['@id']
                    if field_id in df.columns:
                        numeric_field_id = field_id
                        break

    # Fall back to first available numeric column
    if not numeric_field_id:
        for col in df.columns:
            # Try auto-detect numeric
            if pd.api.types.is_numeric_dtype(df[col]):
                numeric_field_id = col
                break

    if not numeric_field_id:
        print('No numeric field found in this record set.')
    else:
        print(f"Numeric field selected (by @id): {numeric_field_id}")

        # Example: Filter for values above a threshold
        threshold = 10
        numeric_values = pd.to_numeric(df[numeric_field_id], errors='coerce')
        filtered_df = df[numeric_values > threshold].copy()
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        display(filtered_df.head())

        # Normalize the selected numeric column
        filtered_values = pd.to_numeric(filtered_df[numeric_field_id], errors='coerce')
        filtered_df[f"{numeric_field_id}_normalized"] = (filtered_values - filtered_values.mean()) / filtered_values.std()
        print(f"Normalized {numeric_field_id} for filtered records:")
        display(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Group by a categorical field, if one exists
        group_field_id = None
        if rs_meta and 'field' in rs_meta:
            for field in fields:
                if isinstance(field, dict) and 'dataType' in field:
                    dtype = field['dataType']
                    if dtype == "schema:Text":
                        field_id = field['@id']
                        if field_id in filtered_df.columns:
                            group_field_id = field_id
                            break

        if group_field_id:
            print(f"Grouping by field: {group_field_id}")
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            display(grouped_df.head())
        else:
            print('No group-able (categorical/text) field found for grouping in this record set.')

## 5. Visualization
Visualize the distribution of the selected numeric field and its normalized version. All axes/titles reference their `@id`s for traceability.

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if 'filtered_df' in locals() and not filtered_df.empty and 'numeric_field_id' in locals() and numeric_field_id:
    fig, axes = plt.subplots(1, 2, figsize=(12, 4))
    sns.histplot(pd.to_numeric(filtered_df[numeric_field_id], errors='coerce'), kde=True, ax=axes[0])
    axes[0].set_title(f'Original Distribution: {numeric_field_id}')
    axes[0].set_xlabel(numeric_field_id)

    norm_field = f"{numeric_field_id}_normalized"
    if norm_field in filtered_df:
        sns.histplot(filtered_df[norm_field].dropna(), kde=True, ax=axes[1], color='orange')
        axes[1].set_title(f'Normalized: {norm_field}')
        axes[1].set_xlabel(norm_field)

    plt.tight_layout()
    plt.show()
else:
    print('Nothing to visualize: no filtered or numeric data available.')

## 6. Conclusion
This notebook demonstrated end-to-end exploration of a Croissant-compatible dataset using `mlcroissant`. We loaded dataset metadata, explored its structure, and processed record sets and fields using their `@id`s to ensure reproducibility and interoperability. For more advanced analysis or tailored EDA, extend this notebook by using the comprehensive field and record set metadata provided in the Croissant schema.